# Universal 3D × MiniMax H3 Studio — Google Colab L4 GPU ワンストップ起動

GitHub リポジトリ（`zentakashi/universal-3d-h3-studio`）から自動で環境を構築し、Material Design 3 UI のワンストップ・スタジオを起動します。
- 任意の `.glb` / `.obj` 3D モデルのドラッグ＆ドロップ読込＆自動スケール調整
- 画面比率（`16:9` / `9:16` / `4:3` / `3:4` / `1:1`）・解像度（`720p` / `1080p`）・FPS（`24` / `30` / `60`）対応
- 3D 画面からの参照画像キャプチャ (`<Picture 1..9>`) & 5秒動画録画 (`<Video 1..3>`) + 参照音声 (`<Audio 1..3>`)
- **MiniMax H3 (`Ref2VA` / `FL2VA` / `T2VA`)** による動画＋32kHz ステレオ音声生成
- Web UI 内ボタンまたは最終セルからの安全な `google.colab.runtime.unassign()`

In [ ]:
# 1. GitHub リポジトリ自動クローン + ComfyUI / 依存ライブラリ セットアップ
import os, torch

assert torch.cuda.is_available(), '上部メニュー「ランタイム」→「ランタイムのタイプを変更」から L4 GPU を選択してください。'
cuda_major = int((torch.version.cuda or '12.4').split('.')[0])
QUANT_MODE = 'int8_convrot' if cuda_major >= 13 else 'fp8_scaled'
print(f'GPU: {torch.cuda.get_device_name(0)} | CUDA: {torch.version.cuda} | Quant: {QUANT_MODE}')

STUDIO_DIR = '/content/universal-3d-h3-studio'
if not os.path.exists(STUDIO_DIR):
    !git clone --depth 1 https://github.com/zentakashi/universal-3d-h3-studio.git {STUDIO_DIR}
else:
    !git -C {STUDIO_DIR} pull --ff-only

!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
if not os.path.exists('/content/ComfyUI'):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
!pip install -q -r /content/ComfyUI/requirements.txt -r {STUDIO_DIR}/requirements.txt 'huggingface_hub[hf_xet]' hf_transfer

vhs = '/content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite'
if not os.path.exists(vhs):
    !git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git {vhs}
    !pip install -q -r {vhs}/requirements.txt

In [ ]:
# 2. MiniMax H3 量子化モデル一式ダウンロード（初回のみ数分かかります）
DOWNLOAD_REF2VA = True   # 汎用参照 (画像+動画+音声) — 推奨
DOWNLOAD_FL2VA  = False  # 開始/終了フレーム固定補間も使う場合は True (+21GB)

import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download, list_repo_files

def fetch(repo, rfile, sub):
    d = f'/content/ComfyUI/models/{sub}'; os.makedirs(d, exist_ok=True)
    dst = os.path.join(d, os.path.basename(rfile))
    if not os.path.exists(dst):
        os.symlink(hf_hub_download(repo_id=repo, filename=rfile), dst)
    print('[OK]', dst)

h3 = list_repo_files('Comfy-Org/MiniMax-H3')
fetch('Comfy-Org/MiniMax-H3', 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'text_encoders')
vvae_cands = [f for f in h3 if f.startswith('vae/minimax_h3_video_vae')]
pref = 'int8_convrot' if QUANT_MODE == 'int8_convrot' else 'fp16'
fetch('Comfy-Org/MiniMax-H3', next((f for f in vvae_cands if pref in f), vvae_cands[0]), 'vae')
fetch('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae')

if DOWNLOAD_REF2VA:
    fetch('Comfy-Org/MiniMax-H3', f'diffusion_models/minimax_h3_ref2va_pruned_{QUANT_MODE}.safetensors', 'diffusion_models')
if DOWNLOAD_FL2VA:
    fetch('Comfy-Org/MiniMax-H3', f'diffusion_models/minimax_h3_fl2va_pruned_{QUANT_MODE}.safetensors', 'diffusion_models')

for repo, fn in [
    ('drbaph/MiniMax-H3-Turbo-Lora-ComfyUI', lambda f: 'pruned' in f.lower() and 'hyperflow' in f.lower() and f.endswith('.safetensors')),
    ('lightx2v/Minimax-h3-Turbo', lambda f: 'ref2v' in f.lower() and ('8step' in f.lower() or '4step' in f.lower()) and f.endswith('.safetensors')),
]:
    try:
        cand = next((f for f in list_repo_files(repo) if fn(f)), None)
        if cand: fetch(repo, cand, 'loras'); break
    except Exception: pass

In [ ]:
# 3. ComfyUI + Universal 3D × MiniMax H3 Studio Web UI を起動
import subprocess, time, urllib.request
from google.colab import output

!pkill -f 'ComfyUI/main.py' || true
!pkill -f 'studio_server.py' || true

subprocess.Popen([
    'python3', '/content/ComfyUI/main.py',
    '--listen', '127.0.0.1', '--port', '8188',
    '--normalvram', '--disable-pinned-memory',
], stdout=open('/content/comfyui.log', 'w'), stderr=subprocess.STDOUT)

subprocess.Popen([
    'python3', '/content/universal-3d-h3-studio/server/studio_server.py',
], stdout=open('/content/studio.log', 'w'), stderr=subprocess.STDOUT)

for _ in range(90):
    try:
        urllib.request.urlopen('http://127.0.0.1:7860/api/health', timeout=2)
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        print('Universal 3D x MiniMax H3 Studio (7860) & ComfyUI (8188) ready!')
        break
    except Exception:
        time.sleep(2)

output.serve_kernel_port_as_window(7860, path='/', anchor_text='▶ Universal 3D × MiniMax H3 Studio を開く')

In [ ]:
# 4. 【必須】作業終了後：L4 GPU ランタイムを切断・削除
from google.colab import runtime
runtime.unassign()